# Masked Inference — Step-by-step

Each **Step** cell is independently runnable. Output files land in
`analysis/depth_view/` with a shared timestamp prefix so one capture's
outputs stay grouped together.

Cell state is shared via a module-level `STATE` dict. If a Step complains
about a missing key, run the prerequisite Step(s) first.

- **Step 1** — Capture one frame (closes the ZED immediately after grab).
- **Step 2** — Depth sanity heatmap.
- **Step 3** — DBSCAN clusters + kept/dropped bboxes on the RGB.
- **Step 4a** — YOLO on each kept cluster's crop.
- **Step 4b** — YOLO on the full frame (gated by `RUN_FULL_FRAME_YOLO`).
- **Step 5** — Combined overlay PNG (side-by-side when Step 4b ran).
- **Step 6** — Optional 3D DBSCAN scatter (plotly HTML, gated).


## 1. Imports

In [ ]:
import sys, time
from pathlib import Path
from datetime import datetime

import numpy as np
import cv2
import pyzed.sl as sl
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import MinMaxScaler
from ultralytics import YOLO

PROJECT_ROOT = Path('/Users/jnoael/Mechatronics_Vision_2026')
sys.path.insert(0, str(PROJECT_ROOT))

# Module-level bag shared between step cells. Inspect with `STATE.keys()`.
STATE: dict = {}
print('imports OK')


## 2. Config — the only cell you normally edit

In [ ]:
# ── Model ─────────────────────────────────────────────────────────
WEIGHTS    = PROJECT_ROOT / 'runs_cnn_transfer' / 'v1.1' / 'weights' / 'best.pt'
CONF_THRES = 0.50
IOU_THRES  = 0.50
IMGSZ      = 640
MAX_DET    = 4
CLASSES    = None
AUGMENT    = False
DEVICE     = None    # None=auto; 'cpu' forces CPU (frees GPU for ZED SDK on Jetson)

# ── ZED camera ────────────────────────────────────────────────────
RESOLUTION  = sl.RESOLUTION.HD1080  # ZED X locks to HD1080/HD1200
FPS         = 5
DEPTH_MODE  = sl.DEPTH_MODE.PERFORMANCE   # PERFORMANCE is lightest on GPU
DEPTH_MIN_M = 0.3

# ── Shutter ───────────────────────────────────────────────────────
SHUTTER_MODE = 'single'   # 'single' | 'burst'
BURST_COUNT  = 5
FRAME_INDEX  = 0          # which burst frame to process (0 .. BURST_COUNT-1)

# ── Depth → DBSCAN ───────────────────────────────────────────────
DEPTH_DOWNSAMPLE_STEP  = 10   # every Nth pixel in X,Y before clustering
DBSCAN_EPS             = 0.03
DBSCAN_MIN_SAMPLES     = 20
MIN_CLUSTER_POINTS     = 50
MAX_CLUSTERS_PER_FRAME = 6

# ── Wall filter ──────────────────────────────────────────────────
WALL_DEPTH_M = 5.0

# ── Output ────────────────────────────────────────────────────────
ANALYSIS_DIR = PROJECT_ROOT / 'analysis' / 'depth_view'

# ── Toggles ───────────────────────────────────────────────────────
RUN_FULL_FRAME_YOLO   = False   # Step 4b: YOLO on the whole RGB for A/B comparison
VISUALIZE_CLUSTERS_3D = False   # Step 6: plotly 3D scatter HTML

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print(f'outputs  -> {ANALYSIS_DIR}')
print(f'weights  : {WEIGHTS.name}')
print(f'shutter  : {SHUTTER_MODE}')


## 3. Helpers — ZED capture

In [ ]:
def open_zed() -> sl.Camera:
    zed = sl.Camera()
    init = sl.InitParameters()
    init.camera_resolution      = RESOLUTION
    init.camera_fps             = FPS
    init.depth_mode             = DEPTH_MODE
    init.coordinate_units       = sl.UNIT.METER
    init.depth_minimum_distance = DEPTH_MIN_M
    err = zed.open(init)
    if err != sl.ERROR_CODE.SUCCESS:
        raise RuntimeError(f'ZED open failed: {err}')
    res = zed.get_camera_information().camera_configuration.resolution
    print(f'  ZED opened  {res.width}x{res.height} @ {FPS} FPS  depth={DEPTH_MODE}')
    return zed


def capture_frame_and_depth(zed, runtime, image_mat, depth_mat):
    if zed.grab(runtime) != sl.ERROR_CODE.SUCCESS:
        return None, None
    zed.retrieve_image(image_mat, sl.VIEW.LEFT)
    zed.retrieve_measure(depth_mat, sl.MEASURE.DEPTH)
    rgb = image_mat.get_data()
    if rgb.ndim == 3 and rgb.shape[2] == 4:
        rgb = cv2.cvtColor(rgb, cv2.COLOR_BGRA2BGR)
    rgb   = np.ascontiguousarray(rgb)
    depth = np.ascontiguousarray(depth_mat.get_data())
    return rgb, depth


def _grab(zed, runtime, image_mat, depth_mat, retries=30):
    for _ in range(retries):
        rgb, depth = capture_frame_and_depth(zed, runtime, image_mat, depth_mat)
        if rgb is not None and depth is not None:
            return rgb, depth
    raise RuntimeError('ZED stopped providing frames')


## 4. Helpers — DBSCAN

In [ ]:
def cluster_depth(depth_m, step=None, eps=None, min_samples=None):
    step        = step        if step        is not None else DEPTH_DOWNSAMPLE_STEP
    eps         = eps         if eps         is not None else DBSCAN_EPS
    min_samples = min_samples if min_samples is not None else DBSCAN_MIN_SAMPLES
    small  = depth_m[::step, ::step]
    h, w   = small.shape
    yy, xx = np.mgrid[0:h, 0:w]
    flat   = small.ravel()
    ok     = np.isfinite(flat) & (flat > 0)
    if ok.sum() < min_samples:
        return np.empty((0, 3), dtype=np.float32), np.empty((0,), dtype=np.int64)
    points = np.column_stack([
        xx.ravel()[ok] * step,
        yy.ravel()[ok] * step,
        flat[ok],
    ]).astype(np.float32)
    points_norm = MinMaxScaler().fit_transform(points)
    labels = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1).fit_predict(points_norm)
    return points, labels


def filter_clusters(points, labels,
                    wall_depth_m=None, min_points=None, max_clusters=None):
    wall_depth_m = wall_depth_m if wall_depth_m is not None else WALL_DEPTH_M
    min_points   = min_points   if min_points   is not None else MIN_CLUSTER_POINTS
    max_clusters = max_clusters if max_clusters is not None else MAX_CLUSTERS_PER_FRAME
    kept, dropped = [], []
    for lbl in set(labels.tolist()):
        if lbl == -1:
            continue
        mask = labels == lbl
        n    = int(mask.sum())
        xs, ys, zs = points[mask, 0], points[mask, 1], points[mask, 2]
        median_depth = float(np.median(zs))
        info = dict(label=int(lbl),
                    bbox=(int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())),
                    median_depth=median_depth, n_points=n)
        if n < min_points:
            info['reason'] = 'too_few_points'; dropped.append(info)
        elif median_depth > wall_depth_m:
            info['reason'] = 'wall'; dropped.append(info)
        else:
            kept.append(info)
    kept.sort(key=lambda c: c['n_points'], reverse=True)
    return kept[:max_clusters], dropped


## 5. Helpers — YOLO inference

In [ ]:
PALETTE = [(0, 255, 0), (0, 255, 255), (255, 128, 0), (255, 0, 255),
           (0, 128, 255), (128, 255, 0)]


def load_model(weights=None):
    weights = Path(weights or WEIGHTS)
    if not weights.exists():
        raise FileNotFoundError(f'weights not found: {weights}')
    print(f'  loading {weights}...')
    return YOLO(str(weights))


def predict_on_image(model, image, crop_origin=(0, 0)):
    """Run YOLO on one image. crop_origin remaps output boxes to full-frame coords."""
    device_kw = {} if DEVICE is None else {'device': DEVICE}
    r = model.predict(image, conf=CONF_THRES, iou=IOU_THRES, imgsz=IMGSZ,
                      max_det=MAX_DET, classes=CLASSES, augment=AUGMENT,
                      verbose=False, **device_kw)[0]
    if r.boxes is None or len(r.boxes) == 0:
        return []
    xyxy   = r.boxes.xyxy.cpu().numpy()
    confs  = r.boxes.conf.cpu().numpy()
    clsids = r.boxes.cls.cpu().numpy().astype(int)
    ox, oy = crop_origin
    dets = []
    for bb, cf, k in zip(xyxy, confs, clsids):
        gx1, gy1, gx2, gy2 = bb + np.array([ox, oy, ox, oy])
        dets.append(dict(bbox=(int(gx1), int(gy1), int(gx2), int(gy2)),
                         cls=int(k), cls_name=model.names.get(int(k), str(k)),
                         conf=float(cf)))
    return dets


def infer_on_clusters(model, rgb, clusters, pad=8):
    detections = []
    H, W = rgb.shape[:2]
    for cl in clusters:
        x1, y1, x2, y2 = cl['bbox']
        x1 = max(0, x1 - pad); y1 = max(0, y1 - pad)
        x2 = min(W, x2 + pad); y2 = min(H, y2 + pad)
        if x2 - x1 < 8 or y2 - y1 < 8:
            continue
        dets = predict_on_image(model, rgb[y1:y2, x1:x2], crop_origin=(x1, y1))
        for d in dets:
            d['cluster_depth'] = cl['median_depth']
        detections.extend(dets)
    return detections


## 6. Helpers — drawing / saving

In [ ]:
def draw_cluster_bboxes(img, kept, dropped):
    out = img.copy()
    for c in dropped:
        x1, y1, x2, y2 = c['bbox']
        cv2.rectangle(out, (x1, y1), (x2, y2), (80, 80, 180), 1)  # faded red = wall/noise
    for i, c in enumerate(kept):
        x1, y1, x2, y2 = c['bbox']
        color = PALETTE[i % len(PALETTE)]
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        cv2.putText(out, f"z={c['median_depth']:.1f}m  n={c['n_points']}",
                    (x1, max(y1 - 6, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                    color, 2, cv2.LINE_AA)
    return out


def draw_detections(img, detections):
    out = img.copy()
    for d in detections:
        x1, y1, x2, y2 = d['bbox']
        color = PALETTE[d['cls'] % len(PALETTE)]
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        tag = f"{d['cls_name']} {d['conf']:.2f}"
        if 'cluster_depth' in d:
            tag += f" | {d['cluster_depth']:.2f}m"
        (tw, th), _ = cv2.getTextSize(tag, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
        cv2.rectangle(out, (x1, y1 - th - 8), (x1 + tw + 6, y1), color, -1)
        cv2.putText(out, tag, (x1 + 3, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 2, cv2.LINE_AA)
    return out


def depth_to_overlay(rgb, depth, alpha=0.55):
    d = np.where(np.isfinite(depth), depth, 0.0)
    d = np.clip(d, 0, WALL_DEPTH_M + 2.0)  # clamp for a readable color spread
    d_norm = (d / (d.max() + 1e-8) * 255).astype(np.uint8)
    heat = cv2.applyColorMap(d_norm, cv2.COLORMAP_TURBO)
    return cv2.addWeighted(rgb, 1 - alpha, heat, alpha, 0)


def save_with_ts(img, name):
    """Write img to ANALYSIS_DIR using STATE['timestamp'] so all outputs of one
    capture share a prefix. Returns the saved path."""
    ts = STATE.get('timestamp', datetime.now().strftime('%Y%m%d_%H%M%S'))
    path = ANALYSIS_DIR / f'{ts}_{name}.png'
    cv2.imwrite(str(path), img)
    print(f'  saved {path.name}')
    return path


## Step 1 — Capture

Opens the ZED, grabs one (or `BURST_COUNT`) frames, **closes the ZED** immediately.
Populates `STATE['rgb']` and `STATE['depth']`. Downstream cells run without the
SDK loaded so they don't compete for its memory.


In [ ]:
zed = open_zed()
runtime   = sl.RuntimeParameters()
image_mat = sl.Mat()
depth_mat = sl.Mat()

STATE['timestamp'] = datetime.now().strftime('%Y%m%d_%H%M%S')

try:
    if SHUTTER_MODE == 'single':
        rgb, depth = _grab(zed, runtime, image_mat, depth_mat)
        STATE['rgb'], STATE['depth'] = rgb, depth
        print(f'  single: {rgb.shape} RGB + {depth.shape} depth')
    elif SHUTTER_MODE == 'burst':
        STATE['rgbs'], STATE['depths'] = [], []
        for _ in range(BURST_COUNT):
            rgb, depth = _grab(zed, runtime, image_mat, depth_mat)
            STATE['rgbs'].append(rgb); STATE['depths'].append(depth)
        idx = max(0, min(FRAME_INDEX, BURST_COUNT - 1))
        STATE['rgb'], STATE['depth'] = STATE['rgbs'][idx], STATE['depths'][idx]
        print(f'  burst: {BURST_COUNT} frames captured; using FRAME_INDEX={idx}')
    else:
        raise ValueError(f'SHUTTER_MODE must be "single" or "burst", got {SHUTTER_MODE!r}')
finally:
    zed.close()
    print('  ZED closed — downstream cells run without SDK loaded')


## Step 2 — Depth sanity heatmap *(optional)*

Colormap of `LAST_DEPTH` overlaid on the RGB. Quickly confirms the capture has
valid depth before burning compute on DBSCAN.


In [ ]:
if 'rgb' not in STATE or 'depth' not in STATE:
    print('run Step 1 first.')
else:
    overlay = depth_to_overlay(STATE['rgb'], STATE['depth'])
    save_with_ts(overlay, '01_depth')


## Step 3 — DBSCAN clusters

Clusters the captured depth, annotates the RGB frame with **kept** clusters
(coloured) and **dropped** clusters (faded red = wall or too-small noise).
Re-runnable on its own — change `DBSCAN_EPS`, `WALL_DEPTH_M`, or
`DEPTH_DOWNSAMPLE_STEP` in Cell 2 and rerun this cell without re-capturing.


In [ ]:
if 'rgb' not in STATE:
    print('run Step 1 first.')
else:
    points, labels = cluster_depth(STATE['depth'])
    kept, dropped  = filter_clusters(points, labels)
    STATE.update(points=points, labels=labels,
                 kept_clusters=kept, dropped_clusters=dropped)
    annotated = draw_cluster_bboxes(STATE['rgb'], kept, dropped)
    save_with_ts(annotated, '02_dbscan')
    print(f'  kept={len(kept)}  dropped={len(dropped)}')
    for c in kept:
        print(f"    cluster {c['label']}  n={c['n_points']}  median_depth={c['median_depth']:.2f}m")


## Step 4a — YOLO on cluster crops

Runs the trained YOLO on each kept cluster's 2D bbox (with padding), remapping
detections back to full-frame coordinates. Loads the model once into
`STATE['model']` and reuses it on subsequent reruns.


In [ ]:
if 'rgb' not in STATE or 'kept_clusters' not in STATE:
    print('run Steps 1 and 3 first.')
else:
    if 'model' not in STATE:
        STATE['model'] = load_model()
    dets = infer_on_clusters(STATE['model'], STATE['rgb'], STATE['kept_clusters'])
    STATE['cluster_detections'] = dets
    annotated = draw_detections(STATE['rgb'], dets)
    save_with_ts(annotated, '03_yolo_clusters')
    print(f'  cluster-mode detections: {len(dets)}')
    for d in dets:
        depth = d.get('cluster_depth')
        depth_str = f'{depth:.2f}m' if depth is not None else '—'
        print(f"    {d['cls_name']} conf={d['conf']:.2f} depth={depth_str}")


## Step 4b — YOLO on full frame *(optional)*

Runs YOLO once on the full RGB for A/B comparison against the masked path.
Gated by `RUN_FULL_FRAME_YOLO` in Cell 2 — toggle True only when you want
the extra inference for the research-paper comparison figure.


In [ ]:
if not RUN_FULL_FRAME_YOLO:
    print('RUN_FULL_FRAME_YOLO = False in Cell 2. Toggle True to run the full-frame comparison.')
elif 'rgb' not in STATE:
    print('run Step 1 first.')
else:
    if 'model' not in STATE:
        STATE['model'] = load_model()
    dets = predict_on_image(STATE['model'], STATE['rgb'])
    STATE['full_detections'] = dets
    annotated = draw_detections(STATE['rgb'], dets)
    STATE['full_overlay']    = annotated
    save_with_ts(annotated, '04_yolo_full')
    print(f'  full-frame detections: {len(dets)}')
    for d in dets:
        print(f"    {d['cls_name']} conf={d['conf']:.2f}")


## Step 5 — Combined overlay

One PNG combining RGB + DBSCAN cluster bboxes + YOLO detections from Step 4a.
If Step 4b ran, produces a side-by-side `masked | full_frame` comparison instead.


In [ ]:
if 'rgb' not in STATE:
    print('run Step 1 first.')
else:
    composite = draw_cluster_bboxes(STATE['rgb'],
                                     STATE.get('kept_clusters', []),
                                     STATE.get('dropped_clusters', []))
    composite = draw_detections(composite, STATE.get('cluster_detections', []))

    if RUN_FULL_FRAME_YOLO and 'full_overlay' in STATE:
        stacked = np.concatenate([composite, STATE['full_overlay']], axis=1)
        cv2.putText(stacked, 'MASKED', (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(stacked, 'FULL FRAME',
                    (composite.shape[1] + 20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2, cv2.LINE_AA)
        save_with_ts(stacked, '05_overlay_compare')
    else:
        save_with_ts(composite, '05_overlay')


## Step 6 — 3D DBSCAN scatter *(optional)*

Plotly 3D scatter of the captured point cloud: kept clusters in colour,
dropped clusters as faded grey, noise lighter still. Gated by
`VISUALIZE_CLUSTERS_3D` in Cell 2. Saves an HTML alongside the PNGs so
all outputs of one capture share the same timestamp prefix.


In [ ]:
if not VISUALIZE_CLUSTERS_3D:
    print('VISUALIZE_CLUSTERS_3D = False in Cell 2. Toggle True to render the 3D scatter.')
elif 'points' not in STATE:
    print('run Step 3 first.')
else:
    import plotly.graph_objects as go
    points, labels = STATE['points'], STATE['labels']
    kept_ids = {c['label'] for c in STATE.get('kept_clusters', [])}
    if len(points) == 0:
        print('  point cloud is empty.')
    else:
        pts = MinMaxScaler().fit_transform(points)
        PALETTE_3D = ['#4C6EF5', '#F03E3E', '#37B24D', '#F59F00',
                      '#AE3EC9', '#1098AD', '#D6336C', '#E8590C']
        traces = []
        for lbl in sorted(set(labels.tolist())):
            m = labels == lbl
            p = pts[m]
            if lbl == -1:
                name, color, size, opacity = 'Noise', 'rgba(180,180,180,0.25)', 1.5, 0.25
            elif lbl in kept_ids:
                name, color, size, opacity = f'Cluster {lbl} (kept)', PALETTE_3D[lbl % len(PALETTE_3D)], 3, 0.9
            else:
                name, color, size, opacity = f'Cluster {lbl} (wall)', 'rgba(90,90,90,0.45)', 2, 0.45
            traces.append(go.Scatter3d(
                x=p[:, 0], y=p[:, 1], z=p[:, 2], mode='markers',
                marker=dict(size=size, color=color, opacity=opacity), name=name))
        fig = go.Figure(traces)
        fig.update_layout(
            title=f'DBSCAN  (eps={DBSCAN_EPS}, minPts={DBSCAN_MIN_SAMPLES}, '
                  f'wall>{WALL_DEPTH_M}m, step={DEPTH_DOWNSAMPLE_STEP})',
            scene=dict(xaxis_title='X (pixel)', yaxis_title='Y (pixel)',
                       zaxis_title='depth (m)',
                       aspectmode='manual', aspectratio=dict(x=1.8, y=1.0, z=0.6)),
            legend=dict(itemsizing='constant'),
            margin=dict(l=0, r=0, b=0, t=40))
        ts = STATE.get('timestamp', datetime.now().strftime('%Y%m%d_%H%M%S'))
        out = ANALYSIS_DIR / f'{ts}_06_3d.html'
        fig.write_html(str(out))
        print(f'  saved {out.name}')
